In [ ]:
import sys
import os.path # REMOVE
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.ticker as ticker

mm = 1.0/2.54/10
repo_root_path = join('..','..')
python_modules_paths = (
    join(repo_root_path, 'src', 'python'),
    'python'
)
for python_modules_path in python_modules_paths:
    if python_modules_path not in sys.path:
        sys.path.append(python_modules_path)

from pilot300W.pilot300W import Pilot300W, MW_fluid, Vn_to_M

from specie_properties.common import AW, MW
from foam.common import read_multiple_df
from foam.foamcase import FoamCase
plt.style.use(os.path.join(repo_root_path, 'src', 'python', 'clc.mplstyle'))

MWs = dict()

# Tests
assert((Vn_to_M('air', 7.2) - 0.000144398)/0.000144398 < 1e-4) # AR
assert((Vn_to_M('Ar',  1.4) - 3.87494e-05)/3.87494e-05 < 1e-4) # FR
assert((Vn_to_M('CO2', 0.3) - 9.14754e-06)/9.14754e-06 < 1e-4) # DC1
assert((Vn_to_M('CO2', 0.1) - 3.04918e-06)/3.04918e-06 < 1e-4) # SL

In [ ]:
p = Pilot300W('pilot300Wv2_leakage1', offset=0.005)

In [ ]:
# Checked manually for leakage1
print(p.mdot('inlet_FR'))
print(p.mdot('inlet_AR'))
print(p.mdot('inlet_SL'))
print(p.mdot('inlet_DC1'))
print(p.mdot('inlet_DC2'))

In [ ]:
# create dataframe
d = {
    'inlet_FR':  [],
    'inlet_AR':  [],
    'inlet_DC1': [],
    'inlet_DC2': [],
    'inlet_SL':  [],
    # 'dilution': [],
    # 'leakage': [],
}
inlet_names = list(d.keys())
d['dilution'] = []
d['leakage'] = []


for i in range(1, 9):
    p = Pilot300W(f'pilot300Wv2_leakage{i}', offset=0.005)
    # print(i)
    for k in inlet_names:
        d[k].append(p.__dict__[k])
    d['dilution'].append(p._dilution_outlet().loc[25:].mean())
    d['leakage'].append(p._leakage_outlet().loc[25:].mean())

df_leakage = pd.DataFrame(data=d)

In [ ]:
df_leakage
# df_leakage.to_latex()
# df['inlet_FR']
# df.iloc[0]

## Single-case plot

In [ ]:
p = Pilot300W('pilot300Wv2_leakage2', offset=0.005)

fig, ax = plt.subplots(figsize=[90*mm, 67.5*mm])

# fluctuations are small => averaging is acceptable despite variable dt
dilution = p._dilution_outlet().loc[25:].mean()
(100*p._dilution_outlet().loc[1:]).plot(label=f'dilution (AR -> FR) = {dilution*100:.2f} %')

leakage = p._leakage_outlet().loc[25:].mean()
(100*p._leakage_outlet().loc[1:]).plot(label=f'leakage (FR -> AR) = {leakage*100:.2f} %')

ax.set(
    xlabel = r'$t$, s',
    ylabel = r'Leakage and dilution, %',
    ylim = [0,20]
)
ax.legend()

## Comparison to experiment

In [ ]:
path = os.path.join('validation','Moldenhauer2009_fig3.2{}.csv')

d_base = dict(
    inlet_FR	= 1.4,
    inlet_AR	= 7.2,
    inlet_SL	= 0.1,
    inlet_DC1	= 0.3,
    inlet_DC2	= 0.0
)

def read_exp_leakage(path, X = 'inlet_FR', rnd=2):
    dat = np.loadtxt(path, skiprows=2, delimiter=',')
    l = []
    for testdat in dat:
        # print()
        assert(round(testdat[0],rnd)==round(testdat[2],rnd)==round(testdat[4],rnd))
        d = d_base.copy()
        d[X] = round(testdat[0],rnd)
        d['D']    = testdat[1]
        d['Dtot'] = testdat[3]
        d['L']    = testdat[5]
        l.append(d)
    return l

L1 = read_exp_leakage(path.format('a'), X = 'inlet_FR')
L2 = read_exp_leakage(path.format('b'), X = 'inlet_AR', rnd=1)
L3 = read_exp_leakage(path.format('c'), X = 'inlet_FR')
L4 = read_exp_leakage(path.format('d'), X = 'inlet_DC1')
L5 = read_exp_leakage(path.format('e'), X = 'inlet_SL')
for el in L3:
    el['inlet_AR'] = 5*el['inlet_FR']

Ls = L1 + L2 + L3 + L4 + L5
Llist = [L1, L2, L3, L4, L5]

In [ ]:
fig, axs = plt.subplots(nrows=3,ncols=2, figsize=[160*mm, 160*mm])

def plot_exp(ax, dlist, X, Y, *args, **kwargs):
    Xs = [dat[X] for dat in dlist]
    Ys = [dat[Y] for dat in dlist]
    ax.plot(Xs, Ys, *args, **kwargs)

def plot_sim(ax, df, X, Y, L3=False, *args, **kwargs):
    criterias = d_base.copy()
    cond = True
    if L3:
        # U_AR also changes here as =5*U_FR
        cond = cond & np.isclose(5*df['inlet_FR'], df['inlet_AR'])
        print(cond)
        del criterias['inlet_FR']
        del criterias['inlet_AR']
    else:
        del criterias[X]
    
    for c,v in criterias.items():
        cond = cond & np.isclose(df[c], v)
    relevant_df = df[cond]
    ax.plot(relevant_df[X], 100*relevant_df[Y], *args, **kwargs)
    ax.set(ylim=(-0.5,7))

# # like in the thesis
# style_D    = dict(marker='+', linestyle='none', color='gray', label='Dilution exp')
# style_Dtot = dict(marker='o', linestyle='-', color='blue', label='Total dilution exp')
# style_L    = dict(marker='v', linestyle='-', color='red', label='Leakage exp')

# style_D_sim    = dict(marker='*', linestyle='none', color='gray', ms=8, label='Dilution sim')
# style_Dtot_sim = dict(marker='*', linestyle='none', color='blue', ms=8, label='Total dilution sim')
# style_L_sim    = dict(marker='*', linestyle='none', color='red', ms=8, label='Leakage sim')

# my style
style_D    = dict(marker='o', linestyle='-', color='#E69F00', label='Dilution experimental')
style_Dtot = dict(marker='o', linestyle='-', color='#009E73', label='Total dilution experimental')
style_L    = dict(marker='o', linestyle='-', color='#0072B2', label='Leakage experimental')

style_D_sim    = dict(marker='x', linestyle='none', color=style_D['color'], ms=8, label='Dilution simulated')
style_Dtot_sim = dict(marker='x', linestyle='none', color=style_Dtot['color'], ms=8, label='Total dilution simulated')
style_L_sim    = dict(marker='x', linestyle='none', color=style_L['color'], ms=8, label='Leakage simulated')

ordax = [axs[0][0], axs[0][1], axs[1][0], axs[1][1], axs[2][0]]


plot_exp(axs[0][0], L1, 'inlet_FR', 'D',    **style_D)
plot_exp(axs[0][0], L1, 'inlet_FR', 'L',    **style_L)
plot_sim(axs[0][0], df_leakage, 'inlet_FR', 'dilution',   **style_D_sim)
plot_sim(axs[0][0], df_leakage, 'inlet_FR', 'leakage',    **style_L_sim)
axs[0][0].set_xlabel(r'$\dot{V}_\mathrm{FR}$, $\mathrm{L}_\mathrm{n}$/min')
axs[0][0].set_ylabel('Leakage and dilution, %')

plot_exp(axs[0][1], L2, 'inlet_AR', 'D',    **style_D)
plot_exp(axs[0][1], L2, 'inlet_AR', 'L',    **style_L)
plot_sim(axs[0][1], df_leakage, 'inlet_AR', 'dilution',   **style_D_sim)
plot_sim(axs[0][1], df_leakage, 'inlet_AR', 'leakage',    **style_L_sim)
axs[0][1].set_xlabel(r'$\dot{V}_\mathrm{AR}$, $\mathrm{L}_\mathrm{n}$/min')
axs[0][1].set_ylabel('Leakage and dilution, %')

plot_exp(axs[1][0], L3, 'inlet_FR', 'D',    **style_D)
plot_exp(axs[1][0], L3, 'inlet_FR', 'L',    **style_L)
plot_sim(axs[1][0], df_leakage, 'inlet_FR', 'dilution', L3=True,  **style_D_sim)
plot_sim(axs[1][0], df_leakage, 'inlet_FR', 'leakage', L3=True,   **style_L_sim)
secax = axs[1][0].secondary_xaxis('top', functions=(lambda u: u*5, lambda u: u/5))
# axs10_U_AR = axs[1][0].twiny()
secax.set_xlabel(r'$\dot{V}_\mathrm{AR}$, $\mathrm{L}_\mathrm{n}$/min')
secax.minorticks_off()
axs[1][0].set_xlabel(r'$\dot{V}_\mathrm{FR}$, $\mathrm{L}_\mathrm{n}$/min')
axs[1][0].set_ylabel('Leakage and dilution, %')
# axs[1][0].xaxis.tick_top()

plot_exp(axs[1][1], L4, 'inlet_DC1', 'D',    **style_D)
plot_exp(axs[1][1], L4, 'inlet_DC1', 'L',    **style_L)
plot_sim(axs[1][1], df_leakage, 'inlet_DC1', 'dilution',   **style_D_sim)
plot_sim(axs[1][1], df_leakage, 'inlet_DC1', 'leakage',    **style_L_sim)
axs[1][1].set_xlabel(r'$\dot{V}_\mathrm{DC1}$, $\mathrm{L}_\mathrm{n}$/min')
axs[1][1].set_ylabel('Leakage and dilution, %')

plot_exp(axs[2][0], L5, 'inlet_SL', 'D',    **style_D)
plot_exp(axs[2][0], L5, 'inlet_SL', 'L',    **style_L)
plot_sim(axs[2][0], df_leakage, 'inlet_SL', 'dilution',   **style_D_sim)
plot_sim(axs[2][0], df_leakage, 'inlet_SL', 'leakage',    **style_L_sim)
axs[2][0].set_xlabel(r'$\dot{V}_\mathrm{SL}$, $\mathrm{L}_\mathrm{n}$/min')
axs[2][0].set_ylabel('Leakage and dilution, %')
axs[2][0].set_xticks(np.arange(0.05,0.21,0.05))

fig.delaxes(axs[2][1])
fig.tight_layout()
axs[2][0].legend(loc='center', bbox_to_anchor=(1.6,0.5), frameon=True)
fig.savefig('pilot300W_leakage.pdf')